# Task 1B — Constrained Run 2 : Modélisation du désaccord annotateur (Soft-Label Learning)

**Entrée** : texte brut du tweet + labels bruts des 3 annotateurs  
**Sortie** : label parmi `premise`, `conclusion`, `none` + vecteur de probabilités  

**Idée centrale** : au lieu d'entraîner sur un label unique (vote majoritaire), on entraîne
le modèle à reproduire la *distribution* des votes humains. Le désaccord est un signal,
pas du bruit.

Ce notebook compare deux stratégies d'adaptation :
- **Full Fine-Tuning** avec KL Divergence loss
- **LoRA** avec KL Divergence loss

Métrique d'entraînement : **KL Divergence / Cross-Entropy sur soft labels**  
Métrique d'évaluation principale : **Macro F1** (2 classes fusionnées, officielle)

## 0. Installation des dépendances

In [1]:
# HuggingFace pour les modèles et l'entraînement, PEFT pour LoRA, scikit-learn pour les métriques.

%pip install transformers datasets evaluate peft accelerate scikit-learn -q


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports et configuration globale

Constante spécifique au Run 2 :

- ALPHA = 0.7 : poids de la KL Divergence dans la loss combinée (70% KL + 30% CE)

In [ ]:
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, f1_score

# ── Reproductibilité ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Chemins ───────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data")
TRAIN_FILE = "/data-lachesis/common/propp/DeniseAtzori/notebooks/DeniseAtzori/LoRA/merged_annotations_v2.json"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Paramètres du modèle ──────────────────────────────────────────────────────
MODEL_NAME = "microsoft/deberta-v3-large" # 434 M paramétres
MAX_LENGTH = 128
NUM_LABELS = 3

LABEL2ID = {"premise": 0, "conclusion": 1, "none": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# ── Poids de la loss combinée (voir Section 4) ───────────────────────────────
# ALPHA contrôle l'équilibre entre la KL Divergence (signal de désaccord)
# et la Cross-Entropy standard (signal du label majoritaire).
# ALPHA=1.0 → KL pure ; ALPHA=0.0 → CE pure (équivalent à Constrained Run 1)
# Valeur de départ recommandée : 0.7 (favorise le signal de désaccord)
ALPHA = 0.7

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device utilisé : {DEVICE}")
print(f"Alpha (poids KL vs CE) : {ALPHA}")

Device utilisé : cuda
Alpha (poids KL vs CE) : 0.7


## 2. Chargement et préparation des données

Charge le fichier JSON et construit pour chaque tweet deux informations absentes du Run 1 :

- soft_label : convertit les 3 votes bruts des annotateurs en distribution de probabilités. Exemple : [premise, premise, none] → [0.67, 0.0, 0.33]
- disagreement_score : entropie de Shannon du soft label — vaut 0 si les 3 annotateurs sont d'accord, et log(3) ≈ 1.099 si chacun a voté différemment

In [2]:
def load_dataset_json(filepath: Path):
    """
    Charge le fichier JSON et retourne les splits train/dev.

    Pour le Constrained Run 2, on extrait deux informations supplémentaires :
      - soft_label  : distribution des votes [P(premise), P(conclusion), P(none)]
      - disagreement_score : entropie de Shannon des votes (mesure d'incertitude)

    L'entropie maximale pour 3 classes est log(3) ≈ 1.099.
    Accord total (3/3 même label) → entropie = 0.
    Désaccord maximal (1/1/1) → entropie = log(3).
    """
    with open(filepath, "r", encoding="utf-8") as f:
        raw = json.load(f)

    has_split_field = "split" in raw[0]
    if not has_split_field:
        random.shuffle(raw)
        cut = int(len(raw) * 0.9)
        for i, entry in enumerate(raw):
            entry["split"] = "train" if i < cut else "dev"

    def build_soft_label(annotations):
        """Convertit les votes bruts en distribution de probabilités."""
        counts = np.zeros(NUM_LABELS)
        for ann in annotations:
            idx = LABEL2ID.get(ann["label"], 2)
            counts[idx] += 1
        return (counts / counts.sum()).tolist()

    def shannon_entropy(soft_label):
        """Calcule l'entropie de Shannon du vecteur de soft labels."""
        probs = np.array(soft_label)
        # On évite log(0) avec un masque sur les probabilités non nulles
        nonzero = probs[probs > 0]
        return float(-np.sum(nonzero * np.log(nonzero)))

    train_samples, dev_samples = [], []
    for entry in raw:
        soft = build_soft_label(entry["annotations"])
        sample = {
            "id"                 : entry["id"],
            "text"               : entry["tweet_text"],
            "hard_label"         : LABEL2ID[entry["majority_label"]],
            "soft_label"         : soft,
            "disagreement_score" : shannon_entropy(soft),
            "source"             : entry.get("source", "unknown"),
        }
        if entry["split"] == "train":
            train_samples.append(sample)
        else:
            dev_samples.append(sample)

    print(f"Train : {len(train_samples)} exemples")
    print(f"Dev   : {len(dev_samples)} exemples")
    return train_samples, dev_samples


train_samples, dev_samples = load_dataset_json(TRAIN_FILE)

Train : 1199 exemples
Dev   : 134 exemples


Statistiques descriptives sur le niveau de désaccord dans le corpus d'entraînement, réparties en trois catégories : accord total (entropie = 0), désaccord partiel, désaccord fort (entropie > 0.8). Permet de quantifier combien de tweets sont réellement ambigus — c'est la justification empirique du Run 2.

In [3]:
# ── Analyse du désaccord dans le dataset ─────────────────────────────────────
# Comprendre la distribution du désaccord permet de mieux évaluer l'impact
# du Soft-Label Learning vs Run 1

import matplotlib.pyplot as plt

disagreements = [s["disagreement_score"] for s in train_samples]

# Catégorisation : accord total / désaccord partiel / désaccord fort
n_full_agree    = sum(1 for d in disagreements if d == 0.0)
n_partial       = sum(1 for d in disagreements if 0.0 < d < 0.8)
n_high_disagree = sum(1 for d in disagreements if d >= 0.8)
n_total         = len(disagreements)

print(f"Accord total    (entropie=0)   : {n_full_agree:4d} ({100*n_full_agree/n_total:.1f}%)")
print(f"Désaccord partiel              : {n_partial:4d} ({100*n_partial/n_total:.1f}%)")
print(f"Désaccord fort  (entropie>0.8) : {n_high_disagree:4d} ({100*n_high_disagree/n_total:.1f}%)")
print(f"\nEntropie moyenne : {np.mean(disagreements):.3f} (max théorique : {np.log(3):.3f})")

Accord total    (entropie=0)   :  687 (57.3%)
Désaccord partiel              :  442 (36.9%)
Désaccord fort  (entropie>0.8) :   70 (5.8%)

Entropie moyenne : 0.255 (max théorique : 1.099)


## 3. Dataset PyTorch — avec injection des labels annotateurs comme features

Différence centrale vs Run 1 : la méthode _build_text() enrichit le texte d'entrée en ajoutant les votes des annotateurs directement dans la séquence tokenisée. Le modèle reçoit donc deux signaux simultanément :

1. Le texte du tweet
2. Le résumé du désaccord humain sous forme textuelle

MAX_LENGTH + 20 : on ajoute 20 tokens de marge pour accueillir le suffixe d'annotation sans tronquer le tweet.

In [ ]:
class EnthymemeDatasetRun2(Dataset):
    """
    Dataset pour le Constrained Run 2.

    Différence clé vs Run 1 : les labels bruts des annotateurs sont disponibles
    à l'entraînement. On les exploite de deux manières :

    1. Via les soft_labels dans la loss (KL Divergence)
    2. Via l'injection textuelle dans l'input du modèle :
       le texte enrichi inclut un résumé du désaccord entre annotateurs,
       ce qui permet au modèle d'apprendre que le désaccord lui-même
       est un signal de difficulté de classification.

    Format d'entrée enrichi :
      [tweet_text] [SEP] annotators: premise premise none
    """

    def __init__(self, samples, tokenizer, inject_annotator_labels=True):
        self.samples                 = samples
        self.tokenizer               = tokenizer
        self.inject_annotator_labels = inject_annotator_labels

    def __len__(self):
        return len(self.samples)

    def _build_text(self, sample):
        """
        Construit le texte d'entrée.

        Si inject_annotator_labels=True, on ajoute les votes des annotateurs
        sous forme de texte naturel après le tweet.
        Exemple :
          "Vaccines cause autism. [SEP] annotations: premise premise none"

        Cela encode le désaccord directement dans l'entrée textuelle du modèle.
        """
        text = sample["text"]
        if self.inject_annotator_labels:
            # Reconstitution approximative des votes à partir du soft label
            # (en l'absence des votes individuels dans le dict sample)
            # Si les votes individuels sont disponibles, les utiliser directement
            soft = sample["soft_label"]
            n_ann = 3
            votes = []
            for label_idx, prob in enumerate(soft):
                count = round(prob * n_ann)
                votes.extend([ID2LABEL[label_idx]] * count)
            annotation_str = " ".join(votes)
            text = f"{text} [annotations: {annotation_str}]"
        return text

    def __getitem__(self, idx):
        sample = self.samples[idx]
        text   = self._build_text(sample)

        encoding = self.tokenizer(
            text,
            max_length=MAX_LENGTH + 20,  # marge pour le texte d'annotation
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        item = {
            "input_ids"      : encoding["input_ids"].squeeze(0),
            "attention_mask" : encoding["attention_mask"].squeeze(0),
            "labels"         : torch.tensor(sample["hard_label"],  dtype=torch.long),
            "soft_labels"    : torch.tensor(sample["soft_label"],  dtype=torch.float),
        }

        if "token_type_ids" in encoding:
            item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        return item


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Sezione 3 — entrambi i split senza injection
train_dataset = EnthymemeDatasetRun2(train_samples, tokenizer,
                                      inject_annotator_labels=False)
dev_dataset   = EnthymemeDatasetRun2(dev_samples,   tokenizer,
                                      inject_annotator_labels=False)

# Aperçu d'un exemple avec injection
example_text = train_dataset.samples[0]["text"]
example_enrichi = train_dataset._build_text(train_dataset.samples[0])
print("Texte original  :", example_text)
print("Texte enrichi   :", example_enrichi)
print("Soft label      :", train_dataset.samples[0]["soft_label"])

/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Texte original  : What part of do you vaccine takers not understand. STOP TAKING THAT SHIT. The KILLER VACCINES. JUST STOP. THEY ARE FAKE & Will eventually KILL YOU
Texte enrichi   : What part of do you vaccine takers not understand. STOP TAKING THAT SHIT. The KILLER VACCINES. JUST STOP. THEY ARE FAKE & Will eventually KILL YOU
Soft label      : [0.2, 0.0, 0.8]


## 4. Fonction de loss : KL Divergence + Cross-Entropy combinées

Définit la loss combinée, cœur du Run 2. Deux composantes :

- L_KL : KL Divergence entre la distribution prédite par le modèle et les soft labels — force le modèle à reproduire l'incertitude humaine
- L_CE : Cross-Entropy standard sur le label majoritaire — ancrage sur la réponse consensus, évite la dérive

Le test sur données factices à la fin vérifie que la fonction ne produit pas d'erreur et retourne une valeur numérique cohérente.

In [5]:
def soft_label_loss(logits, hard_labels, soft_labels, alpha=ALPHA):
    """
    Loss combinée pour le Soft-Label Learning.

    L_total = alpha * L_KL + (1 - alpha) * L_CE

    ├── L_KL : KL Divergence entre distribution prédite et soft labels
    │          → force le modèle à apprendre l'incertitude annotateur
    │          → utile sur les cas borderline (désaccord élevé)
    │
    └── L_CE : Cross-Entropy standard sur le label majoritaire
               → ancrage sur la "bonne réponse" consensus
               → évite la dérive sur les cas à accord total

    Paramètres
    ----------
    logits      : [batch, 3] — sorties brutes du modèle
    hard_labels : [batch]    — indices du label majoritaire
    soft_labels : [batch, 3] — distribution des votes annotateurs
    alpha       : float      — poids de la KL Divergence (0.0 à 1.0)
    """

    # ── Loss KL Divergence ───────────────────────────────────────────────────
    # On veut KL(soft_labels || P_modele)
    # F.kl_div attend (log_prédictions, cible) et calcule :
    # KL = Σ cible * (log cible - log prédiction)
    log_probs  = F.log_softmax(logits, dim=-1)
    soft_clamped = soft_labels.clamp(min=1e-8)  # stabilité numérique (évite log(0))
    loss_kl = F.kl_div(log_probs, soft_clamped, reduction="batchmean")

    # ── Loss Cross-Entropy standard ──────────────────────────────────────────
    loss_ce = F.cross_entropy(logits, hard_labels)

    # ── Combinaison pondérée ─────────────────────────────────────────────────
    return alpha * loss_kl + (1.0 - alpha) * loss_ce


# Vérification rapide sur des données factices
_logits      = torch.randn(4, 3)
_hard_labels = torch.tensor([0, 2, 1, 2])
_soft_labels = torch.tensor([[0.67, 0.0, 0.33],
                              [0.0,  0.0, 1.0 ],
                              [0.33, 0.33, 0.33],
                              [0.0,  0.33, 0.67]])

test_loss = soft_label_loss(_logits, _hard_labels, _soft_labels, alpha=0.7)
print(f"Loss test (α=0.7) : {test_loss.item():.4f}  ✓")

Loss test (α=0.7) : 0.8677  ✓


## 5. Métriques d'évaluation

Deux fonctions d'évaluation :

- compute_metrics : identique au Run 1, calcule le F1 macro sur 3 classes et sur 2 classes fusionnées (métrique officielle). Utilisée par le Trainer à chaque fin d'époque.
- evaluate_with_soft_labels : spécifique au Run 2, analyse séparément les cas à fort désaccord (entropie > 0.5). C'est la vérification de l'hypothèse centrale : le modèle est-il meilleur précisément sur les tweets ambigus grâce au soft-label learning ?

In [6]:
def compute_metrics(eval_pred):
    """
    Métriques d'évaluation — identiques au Run 1 pour comparaison directe.
    F1 macro 2-class fusionnées = métrique officielle de classement.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    f1_3 = f1_score(labels, preds, average="macro", zero_division=0)

    def merge(x):
        return 0 if x in [0, 1] else 1

    preds_2  = np.vectorize(merge)(preds)
    labels_2 = np.vectorize(merge)(labels)
    f1_2 = f1_score(labels_2, preds_2, average="macro", zero_division=0)

    return {
        "f1_macro_3class" : round(f1_3, 4),
        "f1_macro_2class" : round(f1_2, 4),
    }


def evaluate_with_soft_labels(model, dataset, device=DEVICE):
    """
    Évaluation complémentaire :
    - Cross-entropy entre prédictions et soft labels
    - Analyse séparée sur cas d'accord total vs cas de désaccord
      → permet de mesurer si le modèle est meilleur précisément sur
        les cas ambigus (hypothèse centrale du Run 2)
    """
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs, all_soft, all_hard = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**kwargs)
            probs   = F.softmax(outputs.logits, dim=-1).cpu()

            all_probs.append(probs)
            all_soft.append(batch["soft_labels"])
            all_hard.append(batch["labels"])

    all_probs = torch.cat(all_probs, dim=0)
    all_soft  = torch.cat(all_soft,  dim=0)
    all_hard  = torch.cat(all_hard,  dim=0).numpy()

    # Cross-entropy globale sur soft labels
    ce_global = -(all_soft * torch.log(all_probs.clamp(min=1e-8))).sum(dim=-1).mean()
    print(f"Cross-entropy soft labels (global) : {ce_global.item():.4f}")

    # Analyse sur les cas à fort désaccord (entropie des soft labels > 0.5)
    soft_np     = all_soft.numpy()
    entropy_arr = np.array([
        -np.sum(p[p > 0] * np.log(p[p > 0]))
        for p in soft_np
    ])
    disagreement_mask = entropy_arr > 0.5

    if disagreement_mask.sum() > 0:
        ce_disagree = -(all_soft[disagreement_mask] *
                        torch.log(all_probs[disagreement_mask].clamp(min=1e-8))
                       ).sum(dim=-1).mean()
        preds_disagree  = all_probs[disagreement_mask].argmax(dim=-1).numpy()
        labels_disagree = all_hard[disagreement_mask]
        f1_disagree = f1_score(labels_disagree, preds_disagree,
                               average="macro", zero_division=0)
        print(f"Cross-entropy soft labels (cas désaccord fort, n={disagreement_mask.sum()}) : "
              f"{ce_disagree.item():.4f}")
        print(f"F1 macro sur cas de désaccord fort : {f1_disagree:.4f}")

    return all_probs.numpy()

## 6A. Approche 1 — Full Fine-Tuning avec KL Divergence

Charge le modèle complet et définit SoftLabelTrainer, sous-classe de Trainer qui remplace la loss standard par soft_label_loss. Le paramètre alpha est passé au constructeur pour permettre de le faire varier sans modifier la classe.

In [7]:
model_fft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

print(f"Paramètres totaux (FFT) : {sum(p.numel() for p in model_fft.parameters()):,}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Paramètres totaux (FFT) : 435,064,835


In [12]:
class SoftLabelTrainer(Trainer):
    """
    Trainer avec loss KL Divergence + Cross-Entropy combinée.

    Surcharge compute_loss() pour utiliser notre fonction soft_label_loss()
    au lieu de la Cross-Entropy standard de HuggingFace.

    Le paramètre `alpha` est passé via le constructeur pour permettre
    une recherche d'hyperparamètres facile.
    """

    def __init__(self, *args, alpha=ALPHA, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels      = inputs.pop("labels")
        soft_labels = inputs.pop("soft_labels")

        # Les soft_labels sont sur CPU par défaut, on les envoie sur le bon device
        soft_labels = soft_labels.to(labels.device)

        outputs = model(**inputs)
        logits  = outputs.logits

        loss = soft_label_loss(
            logits      = logits,
            hard_labels = labels,
            soft_labels = soft_labels,
            alpha       = self.alpha,
        )

        return (loss, outputs) if return_outputs else loss


# ── Configuration de l'entraînement FFT ──────────────────────────────────────
training_args_fft = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR / "run2_fft"),
    num_train_epochs            = 20,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1e-6, # essayé 5e-6 (peu des changements)
    weight_decay                = 0.1,
    warmup_ratio                = 0.15, # Plusieurs étapes pour stabiliser l'injection
    lr_scheduler_type           = "linear",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro_2class",
    greater_is_better           = True,
    logging_steps               = 20,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
    remove_unused_columns = False,
)

trainer_fft = SoftLabelTrainer(
    model           = model_fft,
    args            = training_args_fft,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    alpha           = ALPHA,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Entraînement FFT avec α={ALPHA} (KL={ALPHA}, CE={1-ALPHA})...")
trainer_fft.train()

Entraînement FFT avec α=0.7 (KL=0.7, CE=0.30000000000000004)...


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,0.330100,0.609422,0.425200,0.640400
2,0.305600,0.624782,0.423400,0.648700
3,0.267700,0.638890,0.444000,0.678500
4,0.243400,0.682175,0.397000,0.602200
5,0.240500,0.665269,0.458100,0.706100
6,0.258200,0.670614,0.456000,0.703700
7,0.233400,0.692903,0.428800,0.657600
8,0.229500,0.675794,0.462900,0.714600
9,0.211900,0.684116,0.446900,0.682100
10,0.201000,0.710394,0.441800,0.685500


TrainOutput(global_step=825, training_loss=0.24939437042583118, metrics={'train_runtime': 424.2195, 'train_samples_per_second': 56.527, 'train_steps_per_second': 3.536, 'total_flos': 3552973348151304.0, 'train_loss': 0.24939437042583118, 'epoch': 11.0})

In [9]:
# Vérification : le modèle prédit-il correctement sans injection ?
# On crée un dataset dev SANS injection pour tester les vraies capacités du modèle

dev_dataset_no_inject = EnthymemeDatasetRun2(
    dev_samples,
    tokenizer,
    inject_annotator_labels=False  # ← testo puro, senza annotazioni
)

preds_no_inject = trainer_fft.predict(dev_dataset_no_inject)
pred_labels     = np.argmax(preds_no_inject.predictions, axis=-1)

f1_no_inject = f1_score(
    preds_no_inject.label_ids,
    pred_labels,
    average="macro",
    zero_division=0
)
print(f"F1 3-class SANS injection : {f1_no_inject:.4f}")
print(f"(Si << 0.97 → le modèle exploite l'injection pour tricher)") # si c'est le cas, mettre "False" dans l'injection

F1 3-class SANS injection : 0.2624
(Si << 0.97 → le modèle exploite l'injection pour tricher)


In [13]:
# ── Évaluation FFT ────────────────────────────────────────────────────────────
print("=== Résultats Full Fine-Tuning (Run 2) ===")
results_fft = trainer_fft.evaluate()
print(results_fft)

preds_fft = trainer_fft.predict(dev_dataset)
pred_labels_fft = np.argmax(preds_fft.predictions, axis=-1)
print("\nRapport de classification (FFT, Run 2) :")
print(classification_report(
    preds_fft.label_ids, pred_labels_fft,
    target_names=list(LABEL2ID.keys())
))

# Évaluation spécifique aux cas de désaccord
probs_fft = evaluate_with_soft_labels(model_fft.to(DEVICE), dev_dataset)

=== Résultats Full Fine-Tuning (Run 2) ===


{'eval_loss': 0.6757940053939819, 'eval_f1_macro_3class': 0.4629, 'eval_f1_macro_2class': 0.7146, 'eval_runtime': 0.6873, 'eval_samples_per_second': 194.965, 'eval_steps_per_second': 4.365, 'epoch': 11.0}

Rapport de classification (FFT, Run 2) :


/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

              precision    recall  f1-score   support

     premise       0.54      0.65      0.59        40
  conclusion       0.00      0.00      0.00         7
        none       0.80      0.79      0.80        87

    accuracy                           0.71       134
   macro avg       0.45      0.48      0.46       134
weighted avg       0.68      0.71      0.69       134

Cross-entropy soft labels (global) : 0.9093
Cross-entropy soft labels (cas désaccord fort, n=68) : 1.2549
F1 macro sur cas de désaccord fort : 0.4299


### Recherche alpha optimale

In [14]:
# ── Ablation rigoureuse sur alpha — 4 ré-entraînements complets ──────────────
# Contrairement à l'analyse post-hoc, chaque valeur d'alpha
# donne lieu à un entraînement indépendant depuis le même point de départ.
# Métrique de comparaison : F1 macro 2-class sur le dev set (métrique officielle).

import gc

ALPHA_VALUES   = [0.0, 0.3, 0.5, 1.0]  # alpha=0.7 déjà connu : 0.7037
ABLATION_PATH  = OUTPUT_DIR / "ablation_alpha_results.csv"

# Résultats déjà connus depuis le training principal
ablation_results = [
    {"alpha": 0.7, "f1_2class": 0.7037, "f1_3class": 0.4560, "best_epoch": 2}
]


def run_alpha(alpha_val):
    """
    Entraîne un modèle FFT complet avec la valeur d'alpha spécifiée
    et retourne les métriques du meilleur checkpoint sur le dev set.
    Même configuration que le training principal — seul alpha change.
    """
    print(f"\n{'='*55}")
    print(f"  Alpha = {alpha_val}  (KL={alpha_val}, CE={round(1-alpha_val, 1)})")
    print(f"{'='*55}")

    # Rechargement du modèle depuis le checkpoint pré-entraîné à chaque run
    # pour garantir que les runs sont indépendants (même initialisation)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    args = TrainingArguments(
        output_dir                  = str(OUTPUT_DIR / f"ablation_alpha_{alpha_val}"),
        num_train_epochs            = 20,
        per_device_train_batch_size = 8,
        per_device_eval_batch_size  = 32,
        learning_rate               = 1e-6,
        weight_decay                = 0.1,
        warmup_ratio                = 0.15,
        lr_scheduler_type           = "linear",
        eval_strategy               = "epoch",
        save_strategy               = "no",
        load_best_model_at_end      = False,
        metric_for_best_model       = "f1_macro_2class",
        greater_is_better           = True,
        remove_unused_columns       = False,
        logging_steps               = 50,
        seed                        = SEED,
        fp16                        = torch.cuda.is_available(),
        report_to                   = "none",
        save_total_limit            = 0,
    )

    trainer = SoftLabelTrainer(
        model           = model,
        args            = args,
        train_dataset   = train_dataset,
        eval_dataset    = dev_dataset,
        compute_metrics = compute_metrics,
        alpha           = alpha_val,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=3)],
    )

    trainer.train()
    eval_results = trainer.evaluate()

    # Récupération de l'époque du meilleur checkpoint
    best_epoch = -1
    for log in trainer.state.log_history:
        if log.get("eval_f1_macro_2class", -1) == eval_results.get("eval_f1_macro_2class"):
            best_epoch = int(log.get("epoch", -1))
            break

    row = {
        "alpha"      : alpha_val,
        "f1_2class"  : eval_results.get("eval_f1_macro_2class", -1),
        "f1_3class"  : eval_results.get("eval_f1_macro_3class", -1),
        "best_epoch" : best_epoch,
    }

    # Libération mémoire GPU avant le prochain run
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return row


# ── Exécution avec reprise automatique ───────────────────────────────────────
# Si le fichier existe déjà (run interrompu), on recharge et on reprend
if ABLATION_PATH.exists():
    df_ablation   = pd.read_csv(ABLATION_PATH)
    done_alphas   = set(df_ablation["alpha"].tolist())
    print(f"Reprise détectée : alphas déjà effectués → {sorted(done_alphas)}")
else:
    df_ablation = pd.DataFrame(ablation_results)  # contient déjà alpha=0.7
    done_alphas = {0.7}

remaining = [a for a in ALPHA_VALUES if a not in done_alphas]
print(f"Alphas restants : {remaining}")

for alpha_val in remaining:
    try:
        row         = run_alpha(alpha_val)
        df_ablation = pd.concat(
            [df_ablation, pd.DataFrame([row])], ignore_index=True
        )
        # Sauvegarde incrémentale après chaque run
        df_ablation.to_csv(ABLATION_PATH, index=False)
        print(f"  → f1_2class={row['f1_2class']:.4f}  "
              f"f1_3class={row['f1_3class']:.4f}  "
              f"best_epoch={row['best_epoch']}  [sauvegardé]")
    except Exception as e:
        print(f"  ERREUR alpha={alpha_val} : {e}")
        gc.collect()
        torch.cuda.empty_cache()
        continue


# ── Tableau comparatif final ──────────────────────────────────────────────────
df_ablation = pd.read_csv(ABLATION_PATH).sort_values("alpha")

print("\n=== Ablation alpha — résultats complets ===")
print(df_ablation[["alpha", "f1_2class", "f1_3class", "best_epoch"]].to_string(index=False))

best = df_ablation.loc[df_ablation["f1_2class"].idxmax()]
print(f"\nMeilleur alpha : {best['alpha']}  "
      f"→ F1 2-class={best['f1_2class']:.4f}  "
      f"F1 3-class={best['f1_3class']:.4f}")

Alphas restants : [0.0, 0.3, 0.5, 1.0]

  Alpha = 0.0  (KL=0.0, CE=1.0)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,1.198100,0.796121,0.262400,0.393700
2,0.742500,0.779506,0.412700,0.644800
3,0.633200,0.836994,0.352100,0.512300
4,0.417500,0.803715,0.443700,0.665700
5,0.272200,1.221467,0.407300,0.625600
6,0.205700,1.808922,0.494800,0.650400
7,0.189100,1.817207,0.423400,0.648700


  → f1_2class=0.6487  f1_3class=0.4234  best_epoch=7  [sauvegardé]

  Alpha = 0.3  (KL=0.3, CE=0.7)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


...

Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,0.822900,0.710833,0.262400,0.393700
2,0.645800,0.601197,0.262400,0.393700
3,0.559900,0.600609,0.433500,0.662400
4,0.459600,0.826837,0.413200,0.626400
5,0.363100,0.687364,0.449700,0.698200
6,0.291100,0.784559,0.463400,0.712200
7,0.232500,0.782134,0.439600,0.687900
8,0.226000,0.808882,0.460200,0.708800
9,0.187800,0.847970,0.447900,0.674700


  → f1_2class=0.6747  f1_3class=0.4479  best_epoch=9  [sauvegardé]

  Alpha = 1.0  (KL=1.0, CE=0.0)


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,0.725400,0.528463,0.262400,0.393700
2,0.525800,0.466616,0.413800,0.624700
3,0.410000,0.492408,0.441800,0.664000
4,0.279700,0.556596,0.453800,0.701100
5,0.209500,0.648273,0.458600,0.694700
6,0.185800,0.626533,0.427900,0.642500
7,0.159600,0.544584,0.452500,0.698300


  → f1_2class=0.6983  f1_3class=0.4525  best_epoch=7  [sauvegardé]

=== Ablation alpha — résultats complets ===
 alpha  f1_2class  f1_3class  best_epoch
   0.0     0.6487     0.4234           7
   0.3     0.6901     0.5059           7
   0.5     0.6747     0.4479           9
   0.7     0.7037     0.4560           2
   1.0     0.6983     0.4525           7

Meilleur alpha : 0.7  → F1 2-class=0.7037  F1 3-class=0.4560


## 6B. Approche 2 — LoRA avec KL Divergence

Identique à la Sezione 6A mais avec adaptation LoRA. Deux différences par rapport au FFT :

- Learning rate plus élevé (5e-4 vs 2e-5) : justifié car seul ~1% des paramètres est mis à jour
- Scheduler cosine au lieu de linéaire : descente plus douce, souvent meilleure pour LoRA sur peu d'époques

In [15]:
# ── Chargement du modèle de base + application de LoRA ───────────────────────
base_model_lora = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

lora_config = LoraConfig(
    r               = 32,
    lora_alpha      = 64,
    target_modules  = ["query_proj", "value_proj"],
    lora_dropout    = 0.1,
    bias            = "none",
    task_type       = TaskType.SEQ_CLS,
)

model_lora = get_peft_model(base_model_lora, lora_config)

trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_lora.parameters())
print(f"Paramètres entraînables (LoRA) : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Paramètres entraînables (LoRA) : 3,148,803 / 438,213,638 (0.72%)


In [17]:
# ── Configuration de l'entraînement LoRA ─────────────────────────────────────
training_args_lora = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR / "run2_lora"),
    num_train_epochs            = 20,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 32,
    learning_rate               = 1e-4,
    weight_decay                = 0.1,
    warmup_ratio                = 0.06,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1_macro_2class",
    greater_is_better           = True,
    logging_steps               = 20,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
    remove_unused_columns = False,
)

trainer_lora = SoftLabelTrainer(
    model           = model_lora,
    args            = training_args_lora,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    alpha           = ALPHA,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=4)],
)

print(f"Entraînement LoRA avec α={ALPHA}...")
trainer_lora.train()

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Entraînement LoRA avec α=0.7...


Epoch,Training Loss,Validation Loss,F1 Macro 3class,F1 Macro 2class
1,0.548000,0.615063,0.262400,0.393700
2,0.564000,0.590656,0.431500,0.667600
3,0.541400,0.551952,0.288900,0.427400
4,0.439000,0.547010,0.473800,0.728800
5,0.465700,0.546850,0.433500,0.662400
6,0.458000,0.566578,0.461800,0.710300
7,0.443600,0.546235,0.475600,0.729400
8,0.386900,0.585242,0.427900,0.658700
9,0.347700,0.596445,0.432400,0.649000
10,0.334100,0.609743,0.451300,0.685700


TrainOutput(global_step=825, training_loss=0.4483090215740782, metrics={'train_runtime': 290.2603, 'train_samples_per_second': 82.615, 'train_steps_per_second': 5.168, 'total_flos': 3589851599888400.0, 'train_loss': 0.4483090215740782, 'epoch': 11.0})

In [18]:
# ── Évaluation LoRA ───────────────────────────────────────────────────────────
print("=== Résultats LoRA (Run 2) ===")
results_lora = trainer_lora.evaluate()
print(results_lora)

preds_lora = trainer_lora.predict(dev_dataset)
pred_labels_lora = np.argmax(preds_lora.predictions, axis=-1)
print("\nRapport de classification (LoRA, Run 2) :")
print(classification_report(
    preds_lora.label_ids, pred_labels_lora,
    target_names=list(LABEL2ID.keys())
))

probs_lora = evaluate_with_soft_labels(model_lora.to(DEVICE), dev_dataset)

=== Résultats LoRA (Run 2) ===


{'eval_loss': 0.5462347865104675, 'eval_f1_macro_3class': 0.4756, 'eval_f1_macro_2class': 0.7294, 'eval_runtime': 0.7571, 'eval_samples_per_second': 176.993, 'eval_steps_per_second': 3.963, 'epoch': 11.0}

Rapport de classification (LoRA, Run 2) :
              precision    recall  f1-score   support

     premise       0.53      0.82      0.65        40
  conclusion       0.00      0.00      0.00         7
        none       0.86      0.71      0.78        87

    accuracy                           0.71       134
   macro avg       0.46      0.51      0.48       134
weighted avg       0.72      0.71      0.70       134



/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/data-lachesis/common/propp/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

Cross-entropy soft labels (global) : 0.7720
Cross-entropy soft labels (cas désaccord fort, n=68) : 1.1226
F1 macro sur cas de désaccord fort : 0.4397


## 8. Comparaison Run 1 vs Run 2

Tableau récapitulatif des 4 configurations. Les lignes Run 1 contiennent des placeholders [à remplir depuis Run 1] — à compléter manuellement avec les résultats du premier notebook après exécution.

In [20]:
# ── Tableau comparatif ────────────────────────────────────────────────────────
# Pour compléter ce tableau, copier les résultats du notebook Run 1

comparison = pd.DataFrame([
    {
        "Run"       : "Constrained 1 — FFT",
        "Loss"      : "CE pondérée",
        "Annotations annotateurs" : "Non",
        "F1 2-class": "0.715",
        "F1 3-class": "0.542",
    },
    {
        "Run"       : "Constrained 1 — LoRA",
        "Loss"      : "CE pondérée",
        "Annotations annotateurs" : "Non",
        "F1 2-class": "0.7070",
        "F1 3-class": "0.4603",
    },
    {
        "Run"       : "Constrained 2 — FFT",
        "Loss"      : f"KL(α={ALPHA}) + CE",
        "Annotations annotateurs" : "Oui (soft labels)",
        "F1 2-class": results_fft.get("eval_f1_macro_2class", "N/A"),
        "F1 3-class": results_fft.get("eval_f1_macro_3class", "N/A"),
    },
    {
        "Run"       : "Constrained 2 — LoRA",
        "Loss"      : f"KL(α={ALPHA}) + CE",
        "Annotations annotateurs" : "Oui (soft labels)",
        "F1 2-class": results_lora.get("eval_f1_macro_2class", "N/A"),
        "F1 3-class": results_lora.get("eval_f1_macro_3class", "N/A"),
    },
])

print(comparison.to_string(index=False))

                 Run           Loss Annotations annotateurs F1 2-class F1 3-class
 Constrained 1 — FFT    CE pondérée                     Non      0.715      0.542
Constrained 1 — LoRA    CE pondérée                     Non     0.7070     0.4603
 Constrained 2 — FFT KL(α=0.7) + CE       Oui (soft labels)     0.7146     0.4629
Constrained 2 — LoRA KL(α=0.7) + CE       Oui (soft labels)     0.7294     0.4756


Identique au Run 1. Deux points d'attention notés dans le docstring :

- Sur le test set, les labels annotateurs ne sont pas disponibles → utiliser inject_annotator_labels=False dans EnthymemeDatasetRun2
- Alternative : utiliser les prédictions du Run 1 comme pseudo-annotations — à documenter dans les working notes

## 9. Génération des fichiers de soumission

In [21]:
# ── Chargement du jeu de test (sans annotations) ─────────────────────────────
# Le test set ne contient pas de champ 'annotations' ni 'majority_label' :
# on charge uniquement le texte et l'id de chaque tweet.

TEST_FILE = Path("/data-lachesis/common/propp/DeniseAtzori/notebooks/DeniseAtzori/LoRA/test_v2.json")

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_raw = json.load(f)

# Construction de la liste de samples (sans hard_label ni soft_label)
test_samples = [
    {
        "id"         : entry["id"],
        "text"       : entry["tweet_text"],
        # Placeholders nécessaires pour que EnthymemeDatasetRun2 fonctionne
        "hard_label" : 0,
        "soft_label" : [1/3, 1/3, 1/3],
    }
    for entry in test_raw
]

# inject_annotator_labels=False : le test set n'a pas d'annotations disponibles.
# Le modèle doit generaliser depuis ce qu'il a appris pendant le training,
# sans pouvoir lire les votes directement dans le texte.
test_dataset = EnthymemeDatasetRun2(
    test_samples,
    tokenizer,
    inject_annotator_labels=False  # ← identique aux conditions réelles d'inférence
)

print(f"Test set chargé : {len(test_samples)} tweets")

Test set chargé : 148 tweets


In [22]:
def generate_submission(model, dataset, samples, run_name, device=DEVICE):
    """
    Génère le fichier de soumission officiel au format demandé.

    Chaque ligne contient :
      - id              : identifiant du tweet
      - hard_label      : label prédit (premise / conclusion / none)
      - prob_premise    : probabilité de la classe 'premise'
      - prob_conclusion : probabilité de la classe 'conclusion'
      - prob_none       : probabilité de la classe 'none'
    """
    model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)

    all_probs, all_preds = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            if "token_type_ids" in batch:
                kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**kwargs)
            probs   = F.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds   = probs.argmax(axis=-1)

            all_probs.extend(probs.tolist())
            all_preds.extend(preds.tolist())

    # Construction du DataFrame de soumission
    rows = []
    for i, sample in enumerate(samples):
        rows.append({
            "id"             : sample["id"],
            "hard_label"     : ID2LABEL[all_preds[i]],
            "prob_premise"   : round(all_probs[i][0], 4),
            "prob_conclusion": round(all_probs[i][1], 4),
            "prob_none"      : round(all_probs[i][2], 4),
        })

    df = pd.DataFrame(rows)
    out_path = OUTPUT_DIR / f"submission_run2_{run_name}.csv"
    df.to_csv(out_path, index=False)
    print(f"Fichier de soumission sauvegardé : {out_path}")
    return df


df_fft  = generate_submission(model_fft.to(DEVICE),  test_dataset, test_samples, "fft")
df_lora = generate_submission(model_lora.to(DEVICE), test_dataset, test_samples, "lora")

print("\nAperçu de la soumission FFT (Run 2) :")
print(df_fft.head())

Fichier de soumission sauvegardé : outputs/submission_run2_fft.csv
Fichier de soumission sauvegardé : outputs/submission_run2_lora.csv

Aperçu de la soumission FFT (Run 2) :
   id hard_label  prob_premise  prob_conclusion  prob_none
0   4       none        0.0596           0.0342     0.9062
1  10       none        0.0839           0.0495     0.8667
2  18    premise        0.7552           0.0895     0.1553
3  36       none        0.0147           0.0144     0.9709
4  39    premise        0.8744           0.0603     0.0654


## 10. Notes méthodologiques

### Pourquoi la KL Divergence plutôt que la Cross-Entropy sur soft labels ?
Les deux sont mathématiquement équivalentes à une constante près lorsque la cible
est fixe (les soft labels ne changent pas pendant l'entraînement). La KL Divergence
est préférée ici car elle est conceptuellement plus claire : elle mesure
**à quel point la distribution prédite s'éloigne de la distribution humaine**.

### Injection des labels annotateurs dans le texte
L'injection textuelle (`[annotations: premise premise none]`) est une stratégie
complémentaire à la modification de la loss. Elle permet au modèle d'apprendre
que certains patterns textuels génèrent systématiquement du désaccord.
Alternative : encoder les labels annotateurs comme features numériques supplémentaires
concaténées au CLS token.

### Valeur optimale d'alpha
- `alpha=0.0` → équivalent au Run 1 (baseline)
- `alpha=1.0` → KL pure, risque d'ignorer le signal de label majoritaire
- `alpha=0.7` → point de départ recommandé par le professeur
Utiliser la cellule d'ablation pour identifier la valeur optimale sur le dev set.

### Inférence sur le test set
Le test set **n'a pas de labels annotateurs**. Pour le Run 2, on a deux options :
1. Utiliser le modèle entraîné **sans** injection textuelle (créer un dataset avec `inject_annotator_labels=False`)
2. Utiliser les prédictions du Run 1 comme "pseudo-annotations" à injecter → à documenter dans les working notes